# Regression and Regularised Linear Regression

This notebook demonstrates core regression concepts using PyTorch and the diabetes dataset. It covers feature normalisation, custom linear regression, gradient descent, model evaluation, learning-rate tuning, and regularised polynomial regression.

## Objectives

- Prepare tabular data for regression modelling.
- Implement a linear regression model manually in PyTorch.
- Evaluate model performance using mean squared error.
- Explore the impact of learning rate on optimisation.
- Apply regularisation to reduce overfitting in polynomial regression.

## Dataset

The first section uses the diabetes dataset to predict disease progression from clinical measurements. The final section uses a small synthetic dataset to demonstrate regularised polynomial regression.


In [ ]:
import torch
from torch import nn # Neural networks
from sklearn import model_selection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
from IPython import display

import typing
%matplotlib inline 
#! Displays matplotlib stuff directly in jupyter notebooks

In [ ]:
diabetes_db = pd.read_csv('https://www4.stat.ncsu.edu/~boos/var.select/diabetes.tab.txt', sep='\t', header=0)
# sn.pairplot(diabetes_db)

In [ ]:
diabetes_db.head(10)

The dataset is split into training and test sets. A fixed random seed is used so that results are reproducible and comparable across the notebook.


In [ ]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    diabetes_db.loc[:, diabetes_db.columns != 'Y'],
    diabetes_db['Y'],
    test_size=0.2,
    random_state=42
    )
x_train = torch.from_numpy(X_train.values).float()
x_test = torch.from_numpy(X_test.values).float()

y_train = torch.from_numpy(y_train.values).float()
y_train = y_train.reshape(-1, 1)

y_test = torch.from_numpy(y_test.values).float()
y_test = y_test.reshape(-1, 1)


# 1. Feature Normalisation


The independent variables are measured on different scales, which can affect gradient descent. To make optimisation more stable, each feature is standardised to have zero mean and unit standard deviation.

The standardised value is calculated as:

$$z_i = \frac{x_i - \mu}{\sigma}$$

where $\mu$ is the mean and $\sigma$ is the standard deviation of the feature.


In [ ]:
def norm_set(x: torch.Tensor, mu: torch.Tensor, sigma: torch.Tensor) -> torch.tensor:
  ### your code here
  norm = (x - mu) / sigma
  return norm
  

###your code here

# ^calc mean and std for x_train
mu = torch.mean(x_train, dim=0)
sigma = torch.std(x_train, dim=0)

# ^ normielies
x_train = norm_set(x_train, mu, sigma)
x_test = norm_set(x_test, mu, sigma)


  

# 2. Linear Regression Model


This section implements a simple linear regression model in PyTorch using a custom layer.

The model follows:

$$y = f(x) = w^Tx$$

The aim is to learn the weight vector $w$ that maps the input features to the target value.


In [ ]:
class LinearRegression(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.weight = nn.Parameter(torch.ones(1, num_features), requires_grad=False)

  def forward(self, x):
    y = 0
    ### your code here
    # ^ calc y through matrix multiplication and transpose it
    y = torch.mm(x, self.weight.T) # ! T for transpose!
    return y


A bias term is added by appending a column of ones to the feature matrix.


In [ ]:
# add a feature for bias
x_train = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1)
x_test = torch.cat([x_test, torch.ones(x_test.shape[0], 1)], dim=1)

In [ ]:
## test the custom layer
model = LinearRegression(x_train.shape[1])
prediction = model(x_train)
prediction.shape # the output should be Nx1


# 3. Mean Squared Error


The model is evaluated using mean squared error, which measures the average squared difference between predictions and true values:

$$E(w) = \frac{1}{N} \sum_{i=0}^{N}(f(x_i) - y_i)^2$$


In [ ]:
def mean_squared_error(y_pred: torch.Tensor, y_true: torch.Tensor) -> torch.Tensor:
  ### your code here
  errors = y_true - y_pred # *Calculate errors
  squared_errors = errors ** 2 # *Square the errors
  mse = squared_errors.mean() # *Calculate mean of squared errors
  return mse

In [ ]:
cost = mean_squared_error(y_train, prediction) # ? Parameters should be swapped? Doesn't matter anyways. Same answer.
print(cost)

# 4. Gradient Descent Optimisation


The initial model parameters produce a high error, so gradient descent is used to iteratively update the weights.

The weight update rule is:

$$weight = weight - \alpha \cdot \partial_{weight}$$

where $\alpha$ is the learning rate.


In [ ]:
def gradient_descent_step(model: nn.Module, X: torch.Tensor, y: torch.Tensor, y_pred: torch.Tensor, lr: float) -> None:
  # & paramaters: model, X tensor, Y tensor, y predictions, learning rate
  weight = model.weight
  ### your code here
  N = X.shape[0]  # ^No. of samples
  
  error = y_pred - y # * prediction errors (ypred - ytrue)

  # calculate the partial derivative of the loss function with respect to w
  gradient = (2 / N) * X.T @ error
  # calculate the new values the weights
  weight = weight - lr * gradient.T
  model.weight = nn.Parameter(weight, requires_grad=False)

In [ ]:
cost_lst = list()
model = LinearRegression(x_train.shape[1])
alpha = 0.1
for it in range(100):
  prediction = model(x_train)
  cost = mean_squared_error(y_train, prediction)
  cost_lst.append(cost)
  gradient_descent_step(model, x_train, y_train, prediction, alpha)
# *The most recent iterations of MSE in cost_lst converge to the minimum MSE value



fig, axs = plt.subplots(2, figsize=(8,10))
# !Plot training cost (MSE) vs Iteration
axs[0].plot(list(range(it+1)), cost_lst)

axs[0].set_title("Training Cost Over Iterations")
axs[0].set_xlabel("Iteration")
axs[0].set_ylabel("Cost (MSE)")

# !Plot Actual vs Predicted 
#*NOTE - should be clearly diagonal
axs[1].scatter(prediction, y_train)

axs[1].set_title("Predicted vs. Actual")
axs[1].set_xlabel("Predicted")
axs[1].set_ylabel("Actual")
# ^Space out the subplots because whoever made this is stinky 
plt.tight_layout(pad=1.0)  
plt.show()

model_weights = model.weight
print(f"Model weights:\n{model_weights}")



min_cost = min(cost_lst)
print('Minimum cost: {}'.format(min_cost))

# 5. Interpreting Model Weights


This section interprets the learned model weights and uses the trained model to estimate disease progression for two example patients.

| AGE | SEX | BMI | BP  | S1  | S2    | S3 | S4  | S5     | S6  |
|-----|-----|-----|-----|-----|-------|----|-----|--------|-----|
| 25  | F   | 18  | 79  | 130 | 64.8  | 61 | 2   | 4.1897 | 68  |
| 50  | M   | 28  | 103 | 229 | 162.2 | 60 | 4.5 | 6.107  | 124 |


The learned weights can be interpreted as the estimated contribution of each feature to the predicted disease progression value.

- A positive weight suggests that higher values of that feature increase the predicted target value.
- A negative weight suggests that higher values of that feature decrease the predicted target value.
- The BMI weight is positive and larger in magnitude than the sex weight, suggesting that BMI has a stronger relationship with predicted disease progression in this model.
- The sex weight is negative and smaller in magnitude, suggesting a weaker relationship compared with BMI.

These interpretations are based on the fitted linear model and should be treated as model-based observations rather than causal conclusions.


In [ ]:
### your code here

# !M encoded as 1
# !F encoded as 2

rows = torch.tensor([
    [25, 0, 18, 79, 130, 64.8, 61, 2, 4.1897, 68],  
    [50, 1, 28, 103, 229, 162.2, 60, 4.5, 6.107, 124]  
])

# ^ add bias. since data already has bias then this also probably should have it as well
rows_bias = torch.cat([rows, torch.ones(rows.shape[0], 1)], dim=1)

predicts = model(rows_bias)


for i, predict in enumerate(predicts):
    print(f"Predictions {i+1}: {predict.item():.2f}")




The trained model is then evaluated on the test set to compare training and test error. This helps assess whether the model is underfitting, overfitting, or generalising reasonably well.


In [ ]:
### your code here

# ^ predict with mdoel
test_predictions = model(x_test)

# ^ calc mse for test and for train just use the cost_lst
test_error = mean_squared_error(test_predictions, y_test)
train_error = min(cost_lst)  

print(f"Train Error: {train_error:.2f}")
print(f"Test Error: {test_error.item():.2f}")




A model **underfits** when it is too simple to capture the patterns in the data.

A model **overfits** when it is too complex and starts learning noise from the training data instead of general patterns.

In general:

- High MSE indicates weaker predictive performance.
- Low training MSE is desirable, but if test MSE is much higher, the model may be overfitting.


# 6. Learning Rate Experiments


This section compares several learning rates to observe their effect on optimisation. Very small learning rates can make training slow, while very large learning rates can overshoot the minimum and prevent stable convergence.


In [ ]:


alphas = [0.001, 0.01, 0.1, 0.2, 0.25, 0.0001]

# ^Iterate over alpha, then go over 100 iterations to plot mse cost over iterations
for alpha in alphas:
    model = LinearRegression(x_train.shape[1])
    train_costs = []  
    
    print(f"\nTesting with alpha = {alpha}")
    
    for it in range(100):
        
        train_predictions = model(x_train)
        
        train_cost = mean_squared_error(train_predictions, y_train)
        train_costs.append(train_cost.item())
        
        gradient_descent_step(model, x_train, y_train, train_predictions, alpha)
    
    # ^ calc min cost for train
    min_train_cost = min(train_costs)
    
    # ^ calc cost for test
    test_predictions = model(x_test)
    test_cost = mean_squared_error(test_predictions, y_test).item()
    
    #^ print
    print(f"Alpha: {alpha}, Min Train Cost: {min_train_cost:.2f}, Test Cost: {test_cost:.2f}")
    
    # & plots
    plt.plot(range(1, 101), train_costs, label=f'Alpha = {alpha}')
    

plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.title('Cost vs Iterations')
plt.legend()
plt.show()



The learning rate begins to overshoot at higher values. A suitable learning rate is normally found through experimentation: higher values may speed up convergence, but they also increase the risk of instability; lower values are safer but may require more iterations.


# 7. Regularised Polynomial Regression


## 7.1 Polynomial Feature Expansion


In [ ]:
x = torch.tensor([-0.99768, -0.69574, -0.40373, -0.10236, 0.22024, 0.47742, 0.82229])
y = torch.tensor([2.0885, 1.1646, 0.3287, 0.46013, 0.44808, 0.10013, -0.32952]).reshape(-1, 1)
plt.scatter(x, y)
plt.show()

To fit the non-linear data, a fifth-order polynomial hypothesis is used:

$$
h_{\theta}(x) = \theta_{0}x_{0} + \theta_{1}x_{1} + \theta_{2}x_{1}^{2} + \theta_{3}x_{1}^{3} + \theta_{4}x_{1}^{4} + \theta_{5}x_{1}^{5}
$$

Because the dataset is small and the model is flexible, there is a risk of overfitting. Regularisation is used to penalise large weights and encourage a smoother model.

The regularised cost function is:

$$
J(\theta) = \frac{1}{2m}\left[\sum_{i=1}^{m}(h_{\theta}(x^{(i)}) - y^{(i)})^2 + \lambda \sum_{j=1}^{n}\theta_j^2\right]
$$

The feature matrix is expanded to include $x_1, x_1^2, \ldots, x_1^5$, along with a bias term.


In [ ]:
# ## your code here. please assign it to the python object 'x3'. e.g x3 = .....
# # stack strings together tensors to a new dimension
# # Include the bias term x0
# x0 = torch.ones(x.shape[0], 1)  # Shape [7, 1]
# # Compute powers of x1 from x1^1 to x1^5
# x_powers = torch.stack([x**i for i in range(1, 6)], dim=1)  # Shape [7, 5]

# # Concatenate x0 and x_powers to form x3
# x3 = torch.cat([x0, x_powers], dim=1)  # Shape [7, 6]


## your code here. please assign it to the python object 'x3'. e.g x3 = .....

# ^ get bias and reshape to column
bias_term = torch.ones(x.shape[0], 1)
print(x.shape)
x_column = x.reshape(-1, 1)
print(x_column.shape)
#^ calculate powers
x_power_1 = x_column ** 1
x_power_2 = x_column ** 2
x_power_3 = x_column ** 3
x_power_4 = x_column ** 4
x_power_5 = x_column ** 5

# ^ concat powers and then again with bias
x_powers = torch.cat([x_power_1, x_power_2, x_power_3, x_power_4, x_power_5], dim=1)
x3 = torch.cat([bias_term, x_powers], dim=1)





The cost and gradient descent functions are extended to include regularisation. The bias term is not regularised, so it uses a different update rule from the remaining weights.

For the bias term:

$$
\theta_j = \theta_j - \alpha \frac{1}{m}\sum_{i=1}^{m}(h_{\theta}(x^{(i)}) - y^{(i)})x_j^{(i)}, \quad j=0
$$

For the remaining weights:

$$
\theta_j = \theta_j\left(1 - \alpha\frac{\lambda}{m}\right) - \alpha\frac{1}{m}\sum_{i=1}^{m}(h_{\theta}(x^{(i)}) - y^{(i)})x_j^{(i)}, \quad j>0
$$


In [ ]:
def mean_squared_error2(y_true: torch.Tensor, y_pred: torch.Tensor, lam: float, theta: torch.tensor) -> torch.Tensor:
    ### your code here
    #^ no. of samples
    m = y_true.shape[0]  
    # ^ error
    error = y_pred - y_true  
    
    # ^mse
    mse = (1 / (2 * m)) * torch.sum(error ** 2)

    # ^calc regularized term (exclude theta_0 because this is the bias)
    reg_term = (lam / (2 * m)) * torch.sum(theta[0, 1:] ** 2)

    
    cost = mse + reg_term
    return cost
  

def gradient_descent_step2(model: nn.Module, X: torch.Tensor, y: torch.Tensor, y_pred: torch.Tensor, lr: float, lam: float) -> None:
    weight = model.weight
    N = X.shape[0]
    
    # Calculate gradients for each weight
    errors = y_pred - y
    gradients = (1 / N) * X.T @ errors

    # Update bias weight (without regularization)
    weight[0, 0] -= lr * gradients[0, 0]
    
    # Update other weights (with regularization)
    regularization_term = (lam / N) * weight[0, 1:]
    weight_update = lr * (gradients[1:].T + regularization_term)
    weight[0, 1:] -= weight_update.squeeze()  # Use squeeze to ensure correct shape alignment
    
    
    model.weight = nn.Parameter(weight, requires_grad=False)


# 8. Tuning Alpha and Lambda


This section experiments with different values of the learning rate, alpha, and the regularisation strength, lambda. The aim is to observe how these hyperparameters affect convergence and the shape of the fitted polynomial curve.


In [ ]:

# ! Code for finding best alpha
alphas = [0.001, 0.01, 0.1, 0.5, 0.6, 0.7, 0.8]


lam = 0


plt.figure(figsize=(10, 6))

# ^ go over each alpha
for alpha in alphas:
    
    model = LinearRegression(x3.shape[1])
    train_costs = []  
    
    print(f"\nTesting with alpha = {alpha}")
    
    for it in range(100):
        train_predictions = model(x3)
        train_cost = mean_squared_error(y, train_predictions)
        train_costs.append(train_cost.item())
        gradient_descent_step(model, x3, y, train_predictions, alpha)
    
    # ^ calc min train error
    min_train_cost = min(train_costs)
    
    # ^ calc test error
    test_predictions = model(x3)
    test_cost = mean_squared_error(y, test_predictions).item()
    

    print(f"Alpha: {alpha}, Min Train Cost: {min_train_cost:.2f}, Test Cost: {test_cost:.2f}")
    
    # ^ plot cost over iterations
    plt.plot(range(1, 101), train_costs, label=f'Alpha = {alpha}')
    

plt.xlabel('Iteration')
plt.ylabel('Training Cost (MSE)')
plt.title('Training Cost over Iterations for Different Alpha Values')
plt.legend()
plt.grid()
plt.show()





In [ ]:

# ! Plotting hypothesis for each alpha
for alpha in alphas:
    
    model = LinearRegression(x3.shape[1])
    for it in range(100):
        train_predictions = model(x3)
        gradient_descent_step(model, x3, y, train_predictions, alpha)
    
    # ^plot data fit for each alpha 
    plt.figure(figsize=(8, 5))
    #^ make sure to exclude bias from x3 to plot
    plt.scatter(x3[:, 1], y, color='red', label='Data')  
    
    plt.plot(x3[:, 1], model(x3).detach(), label=f'Alpha = {alpha}')  
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'For Alpha = {alpha}')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:

#! For finding best lambda
lambdas = [0, 0.01, 0.1, 1, 2, 3, 4, 10]

#^ best alpha from before
alpha =0.7


plt.figure(figsize=(8, 4))

# ^go over each lambda 
for lam in lambdas:

    model = LinearRegression(x3.shape[1])
    train_costs = []  
    
    print(f"\nTesting with lambda = {lam}")
    

    for it in range(100):

        train_predictions = model(x3)
        
        train_cost = mean_squared_error2(y, train_predictions, lam, model.weight)
        train_costs.append(train_cost.item())
        
        gradient_descent_step2(model, x3, y, train_predictions, alpha, lam)
    
    # ^ calc min train error
    min_train_cost = min(train_costs)
    
    # ^calc test cost
    test_predictions = model(x3)
    test_cost = mean_squared_error2(y, test_predictions, lam, model.weight).item()

    print(f"Lambda: {lam}, Min Train Cost: {min_train_cost:.2f}, Test Cost: {test_cost:.2f}")
    
    # ^ plot cost over iterations
    plt.plot(range(1, 101), train_costs, label=f'Lambda = {lam}')
    

plt.xlabel('Iteration')
plt.ylabel('Training Cost (MSE)')
plt.title('Training Cost vs Iterations ')
plt.legend()
plt.grid()
plt.show()



In [ ]:

# ! plot hypothesis for each lambda
for lam in lambdas:
    
    model = LinearRegression(x3.shape[1])
    for it in range(100):
        train_predictions = model(x3)
        gradient_descent_step2(model, x3, y, train_predictions, alpha, lam)
    
    
    plt.figure(figsize=(6, 4))
    #^ make sure to exclude bias from x3 to plot
    plt.scatter(x3[:, 1], y, color='red', label='Data')  
    plt.plot(x3[:, 1], model(x3).detach(), label=f'Lambda = {lam}')  
    
    plt.xlabel('x')
    plt.ylabel('y')
    plt.title(f'Model Fit for Lambda = {lam}')
    plt.legend()
    plt.grid()
    plt.show()
